# T1-bonus · Knowledge as code

## Goal

Move the Bicep for Search + Storage and the knowledge-source YAML from
one-off notebook cells into a reproducible `pac copilot push` sync, and
prove idempotency: apply twice, assert exactly one knowledge source exists
per definition, not two.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")
assert (workspace / "knowledge").exists()
sources = list((workspace / "knowledge").glob("*.yaml"))
print(f"{len(sources)} knowledge source definitions found")


## Concept

Notebooks `03`-`06` built knowledge sources interactively, one cell at a
time. That's right for learning; it's wrong for a repo you want a second
engineer to trust. This notebook is the "now do it as CI would" pass: Bicep
for the Azure deps, `pac copilot push` for the source definitions, and a
concrete idempotency test rather than a promise.


## Build


In [ ]:
import subprocess
deploy = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", "$AZURE_RESOURCE_GROUP",
    "--template-file", "../infra/bicep/main.bicep",
    "--parameters", "namePrefix=crd-dev", "location=$AZURE_LOCATION",
], capture_output=True, text=True)
print("first apply:", deploy.returncode)


In [ ]:
from csx.pac import copilot_push
from pathlib import Path
copilot_push(Path("../agents/contract-renewal-desk"))


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess, json

# Re-apply Bicep — Azure's declarative model means a no-op re-run creates nothing new.
deploy2 = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", "$AZURE_RESOURCE_GROUP",
    "--template-file", "../infra/bicep/main.bicep",
    "--parameters", "namePrefix=crd-dev", "location=$AZURE_LOCATION",
], capture_output=True, text=True)

result = subprocess.run(["pac", "copilot", "knowledge", "list",
                           "--name", "crd_contract-renewal-desk", "--json"],
                          capture_output=True, text=True)
sources = json.loads(result.stdout) if result.returncode == 0 else []
names = [s["id"] for s in sources]
assert len(names) == len(set(names)), f"duplicate knowledge sources after re-apply: {names}"
print(f"idempotency proven: {len(names)} unique sources after two applies")


## Cost


In [ ]:
print("Bicep re-apply of existing resources: no new spend. Knowledge source re-push: no new spend (config sync, not a build event).")


## Teardown


In [ ]:
print("No teardown — this notebook only proves the pipeline pattern used by every prior Track 1 notebook.")
